# ISM 6564 · Week 6 · Run your fine-tuned model with Ollama (optional)

**ISM 6564 — Text Analytics · Fall 2026 · Week 6: Prompting, Frontier APIs & Fine-Tuning**

## About this notebook

- **Optional and ungraded.** Nothing in Assignment 6 depends on it.
- **What it does.** It takes the LoRA adapter *you* trained in Assignment 6, folds it into the
  base model, converts the result into one file that Ollama can run, and then calls your model
  through the same OpenAI-style API the lecture used — on your own machine, with no API key and no
  cost.
- **Where to run it.** On your own computer (with the course Python environment) or on Google
  Colab. A CPU is enough: nothing here trains a model.
- **What you need first.** The adapter folder `ism6564-week06-adapter/` (about 34 MB), saved by the
  optional cell at the end of Assignment 6, Task 4. On Colab that cell also gives you a zip to
  download. To *run* the model at the end you need Ollama installed on your own computer:
  [ollama.com/download](https://ollama.com/download) (Windows, macOS or Linux). Tested with Ollama
  0.34.1.
- **Time and space.** About two minutes the first time (the base model and the converter are
  downloaded once), and about 2.5 GB of free disk while it works; the finished file is about 530 MB.
- **Support.** As with every install in this course, getting Ollama and your own machine working is
  your job; there is no tech support for personal installs.

**The route, in four steps:** adapter → merged model → one GGUF file (the format Ollama runs) →
Ollama. The middle step is needed because Ollama cannot read this model family straight from
Hugging Face's format.


## Step 1 — Find your adapter

Put `ism6564-week06-adapter/` (or `ism6564-week06-adapter.zip`) in the same folder as this notebook.
On Colab, the cell below asks you to upload the zip if it cannot find either.


In [ ]:
import importlib.util
import os
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

for mod, pkg in (("peft", "peft"), ("transformers", "transformers")):
    if importlib.util.find_spec(mod) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

IN_COLAB = "google.colab" in sys.modules
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"           # the model Assignment 6 fine-tuned
ADAPTER = Path("ism6564-week06-adapter")
ZIP = Path("ism6564-week06-adapter.zip")

if not ADAPTER.exists() and not ZIP.exists() and IN_COLAB:
    from google.colab import files
    print("Upload ism6564-week06-adapter.zip (from Assignment 6):")
    files.upload()
if not ADAPTER.exists() and ZIP.exists():
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(ADAPTER)
assert (ADAPTER / "adapter_config.json").exists(), (
    f"No adapter found. Run the optional save cell at the end of Assignment 6, Task 4, and put "
    f"'{ADAPTER}/' or '{ZIP}' next to this notebook.")
size = sum(f.stat().st_size for f in ADAPTER.rglob("*") if f.is_file())
print(f"adapter found: {ADAPTER}/ ({size / 1e6:.0f} MB)")


## Step 2 — Fold the adapter into the model

LoRA keeps the base model frozen and learns a small change beside it (lecture: *LoRA: freeze W,
learn two thin tables*). To give Ollama one ordinary model, we add that change into the weights
once — `merge_and_unload()` — and save the result at half precision, the size Ollama expects.


In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MERGED = Path("ism6564-week06-merged")
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL).float()      # CPU is fine: nothing trains here
merged = PeftModel.from_pretrained(base, str(ADAPTER)).merge_and_unload()
merged.to(torch.float16).save_pretrained(MERGED, safe_serialization=True)
tok.save_pretrained(MERGED)
size = sum(f.stat().st_size for f in MERGED.rglob("*") if f.is_file())
print(f"merged model saved: {MERGED}/ ({size / 1e9:.2f} GB)")


## Step 3 — Convert it to one GGUF file

GGUF is the single-file format Ollama runs. The converter comes from the open-source llama.cpp
project; this cell downloads the exact version this notebook was tested with (a 40 MB zip, once)
and stores the weights at 8 bits per number (`q8_0`), which keeps the file near 530 MB with little
change to the answers.


In [ ]:
import urllib.request

COMMIT = "6c7a87f7e5e5cd75b8a641c3471f2dee84a6ed17"
TOOL = Path(f"llama.cpp-{COMMIT}")
if not TOOL.exists():
    url = f"https://github.com/ggml-org/llama.cpp/archive/{COMMIT}.zip"
    print("downloading the converter ...")
    urllib.request.urlretrieve(url, "llama.cpp.zip")
    with zipfile.ZipFile("llama.cpp.zip") as z:
        z.extractall(".")
    os.remove("llama.cpp.zip")

GGUF = Path("ism6564-week06-triage-q8_0.gguf")
result = subprocess.run([sys.executable, str(TOOL / "convert_hf_to_gguf.py"), str(MERGED),
                         "--outfile", str(GGUF), "--outtype", "q8_0"], capture_output=True, text=True)
if result.returncode != 0:
    print(result.stderr[-2000:])
    raise SystemExit("conversion failed -- see the message above")
Path("Modelfile").write_text(f"FROM ./{GGUF.name}\nPARAMETER temperature 0\n")
print(f"{GGUF} ({GGUF.stat().st_size / 1e6:.0f} MB) and Modelfile written")


## Step 4 — Put it into Ollama

A **Modelfile** is Ollama's recipe: here, one line saying which file to load, and greedy decoding
(temperature 0) so the same ticket always gets the same answer.

- **On your own computer with Ollama installed**, the next cell does it for you.
- **On Colab**, the next cell downloads the GGUF file and the Modelfile. Put both in one folder on
  your computer, open a terminal there, and run:

  ```
  ollama create ism6564-w6-triage -f Modelfile
  ```

  If the 530 MB browser download fails, copy the file to Google Drive instead (Files panel →
  mount Drive → drag it into `MyDrive`) and download it from Drive.


In [ ]:
MODEL_NAME = "ism6564-w6-triage"
if IN_COLAB:
    from google.colab import files
    files.download(str(GGUF))
    files.download("Modelfile")
    print(f"Now, on your computer: ollama create {MODEL_NAME} -f Modelfile")
elif shutil.which("ollama"):
    out = subprocess.run(["ollama", "create", MODEL_NAME, "-f", "Modelfile"], capture_output=True, text=True)
    print(f"created the Ollama model '{MODEL_NAME}'" if out.returncode == 0 else re.sub(r"\x1b\[[0-9;?]*[A-Za-z]", "", out.stderr)[-600:])
else:
    print(f"Ollama is not installed here. Install it from https://ollama.com/download, then run in this folder:\n"
          f"    ollama create {MODEL_NAME} -f Modelfile")


## Step 5 — Call your model through an API

Ollama serves every model on your machine at `http://localhost:11434`, and it speaks the same API
shape as OpenAI. So the lecture's code works unchanged except for two settings: a `base_url` that
points at your machine, and an `api_key` that can be any text (nothing is billed). This cell sends
20 held-out tickets — the start of Assignment 6's evaluation set — to *your* model and scores them.


In [ ]:
import json
import re
import urllib.request

import pandas as pd

URL = "https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/week-06-prompting-apis-finetuning/data"
tickets = pd.read_csv(f"{URL}/support_tickets_test.csv", dtype=str).fillna("").iloc[40:60]
CODES = ["SHIP-DELAY", "RET-RMA", "BILL-DISP", "PROD-DEF", "ACCT-SEC", "PROD-INFO"]
SYSTEM = ("You triage support tickets for Northwind Outfitters.\n"
          'Reply with JSON: {"route_to": one of ' + "|".join(CODES) +
          ', "urgency": low|normal|high, "needs_human": true|false, "order_id": string or ""}')   # Assignment 6's minimal prompt


def ask(ticket: str) -> str:
    """One call to your model through Ollama's OpenAI-compatible API."""
    from openai import OpenAI
    client = OpenAI(base_url="http://localhost:11434/v1/", api_key="ollama")   # the key is required but ignored
    r = client.chat.completions.create(model=MODEL_NAME, temperature=0, max_tokens=80,
                                       messages=[{"role": "system", "content": SYSTEM},
                                                 {"role": "user", "content": f"<ticket>\n{ticket}\n</ticket>"}])
    return r.choices[0].message.content


def last_json(raw: str) -> dict:
    for c in reversed(re.findall(r"\{.*?\}", raw, re.S)):
        try:
            return json.loads(c)
        except json.JSONDecodeError:
            pass
    return {}


try:
    urllib.request.urlopen("http://localhost:11434/api/version", timeout=3)
    running = True
except OSError:
    running = False

if IN_COLAB or not running:
    print("Ollama is not running on this machine -- run this cell on your own computer after Step 4.")
else:
    if importlib.util.find_spec("openai") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openai"])
    right = 0
    for i, (_, t) in enumerate(tickets.iterrows()):
        d = last_json(ask(t.text))
        ok = (d.get("route_to") == t.category and d.get("urgency") == t.urgency
              and str(d.get("needs_human")).lower() == t.needs_human and (d.get("order_id") or "") == t.order_id)
        right += ok
        if i == 0:
            print("first reply:", json.dumps(d))
    print(f"all four fields right: {right} of {len(tickets)}")


## What you just did

- You took an adapter you trained, merged it into its base model, packaged it as one file, and ran
  it on your own machine behind an API — the same steps a company takes to serve a fine-tuned model
  on its own hardware, so no customer text leaves the building (lecture: *What the shorter prompt is
  worth*).
- **Expect close, not identical, numbers.** The Ollama copy stores each number in 8 bits and runs on
  a different engine, so a few tickets can come out differently from your notebook. On the
  instructor's run, 68 of 80 answers were identical and the all-four-right count moved by a few
  tickets.
- Any program can now use your model: point an OpenAI client at `http://localhost:11434/v1/` with
  `model="ism6564-w6-triage"`. To remove it: `ollama rm ism6564-w6-triage`.
